# A4: Do you AGREE

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [3]:
# ===== FORCE USE FREE GPU (GPU 1) =====
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # use GPU 1 (free)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ===== IMPORTS =====
import math
import re
from random import *
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from datasets import load_dataset
from tqdm.auto import tqdm

# ===== CUDA DETECTION =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("GPU Index:", torch.cuda.current_device())
else:
    print("⚠️ Running on CPU")

# ===== REPRODUCIBILITY =====
SEED = 1234
torch.manual_seed(SEED)

if device.type == "cuda":
    torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True


Using device: cuda
✅ GPU: NVIDIA GeForce RTX 2080 Ti
GPU Index: 0


# Task 1

### LOADING DATASET



The model is trained using a subset of the BookCorpus dataset obtained from Hugging Face:

Dataset source:
SamuelYang/bookcorpus  
https://huggingface.co/datasets/SamuelYang/bookcorpus

tHE BookCorpus contains a large collection of English text suitable for language model pre-training.

Due to computational limitations, only a subset of 100,000 samples was used instead of the full corpus, as allowed by the assignment instructions.


In [4]:
from datasets import load_dataset
import re

SEED = 1234

ds = load_dataset("SamuelYang/bookcorpus")

split_ds = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds = split_ds["train"].select(range(100000))

print("Training samples:", len(train_ds))

sentences = train_ds["text"]

# preprocess
text = [s.lower() for s in sentences]
text = [re.sub("[.,!?\\-]", "", s) for s in text]


Training samples: 100000


## PREPROCESS TEXT


The dataset text is preprocessed by:

• Converting text to lowercase  
• Removing punctuation  
• Tokenizing sentences into words  
• Building a vocabulary from the dataset  

Special tokens used:

[PAD]  — padding token  
[CLS]  — classification token  
[SEP]  — sentence separator  
[MASK] — masked token for MLM training  

Each sentence is converted into token IDs using the constructed vocabulary.


In [5]:
from tqdm.auto import tqdm

word_list = list(set(" ".join(text).split()))

word2id = {'[PAD]':0,'[CLS]':1,'[SEP]':2,'[MASK]':3}

for i, w in tqdm(enumerate(word_list)):
    word2id[w] = i + 4

id2word = {v:k for k,v in word2id.items()}
vocab_size = len(word2id)

token_list = []
for sentence in tqdm(text):
    token_list.append([word2id[w] for w in sentence.split()])


0it [00:00, ?it/s]

  0%|          | 0/100000 [00:00<?, ?it/s]

## MAKE BATCH (MLM + NSP)


### 1. Masked Language Modeling (MLM)
Randomly masks 15% of tokens in the input and trains the model to predict the original tokens.

Masking strategy:
• 80% replaced with [MASK]
• 10% replaced with random token
• 10% unchanged

### 2. Next Sentence Prediction (NSP)
Determines whether two sentences appear consecutively in the original text.

Label:
* 1 → sentence B follows sentence A  
* 0 → sentence B is random  

Both objectives help the model learn deep bidirectional context.


In [6]:
from random import *

batch_size = 6
max_mask = 5
max_len = 128

def make_batch():
    batch = []
    pos = neg = 0

    while pos != batch_size//2 or neg != batch_size//2:

        a, b = randrange(len(token_list)), randrange(len(token_list))
        tokens_a, tokens_b = token_list[a], token_list[b]

        input_ids = [word2id['[CLS]']] + tokens_a + [word2id['[SEP]']] + tokens_b + [word2id['[SEP]']]
        segment_ids = [0]*(len(tokens_a)+2) + [1]*(len(tokens_b)+1)

        n_pred = min(max_mask, max(1,int(len(input_ids)*0.15)))

        cand = [i for i in range(len(input_ids))
                if input_ids[i] not in [word2id['[CLS]'],word2id['[SEP]']]]
        shuffle(cand)

        masked_tokens, masked_pos = [], []

        for p in cand[:n_pred]:
            masked_pos.append(p)
            masked_tokens.append(input_ids[p])

            r = random()
            if r < 0.8:
                input_ids[p] = word2id['[MASK]']
            elif r < 0.9:
                input_ids[p] = randint(4, vocab_size-1)

        input_ids = input_ids[:max_len]
        segment_ids = segment_ids[:max_len]

        input_ids += [0]*(max_len-len(input_ids))
        segment_ids += [0]*(max_len-len(segment_ids))

        masked_tokens += [0]*(max_mask-n_pred)
        masked_pos += [0]*(max_mask-n_pred)

        if a+1 == b and pos < batch_size//2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, 1])
            pos += 1
        elif a+1 != b and neg < batch_size//2:
            batch.append([input_ids, segment_ids, masked_tokens, masked_pos, 0])
            neg += 1

    return batch


In [7]:
def get_attn_pad_mask(seq):
    return seq.eq(0).unsqueeze(1).expand(seq.size(0), seq.size(1), seq.size(1))


## Embedding

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class Embedding(nn.Module):
    def __init__(self, vocab_size, max_len, d_model):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.seg = nn.Embedding(2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, seg):
        pos = torch.arange(x.size(1), device=device).unsqueeze(0)
        return self.norm(self.tok(x) + self.pos(pos) + self.seg(seg))


In [9]:
class ScaledDotAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.scale = torch.sqrt(torch.FloatTensor([d_k])).to(device)

    def forward(self, Q, K, V, mask):
        scores = torch.matmul(Q, K.transpose(-1,-2)) / self.scale
        scores.masked_fill_(mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        return context


## Multi-Head Attention + FFN

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        batch_size = x.size(0)

        Q = self.W_Q(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
        K = self.W_K(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)
        V = self.W_V(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)

        mask = mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)

        context = ScaledDotAttention(self.d_k)(Q,K,V,mask)
        context = context.transpose(1,2).contiguous().view(batch_size, -1, self.n_heads*self.d_k)

        return self.fc(context)


## ENCODER LAYER

In [11]:
# ENCODER LAYER
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.GELU(),
            nn.Linear(d_model*4, d_model)
        )

    def forward(self, x, mask):
        x = x + self.attn(x, mask)
        x = x + self.ff(x)
        return x


## BERT


BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based language model that learns contextual representations of text by jointly conditioning on both left and right context.

Key components implemented from scratch:

• Token, positional, and segment embeddings  
• Multi-head self-attention mechanism  
• Transformer encoder layers  
• Feed-forward networks  
• Masked Language Modeling (MLM) head  
• Next Sentence Prediction (NSP) head  

This implementation follows the concepts taught in class without using pre-trained models or high-level libraries such as HuggingFace Transformers.


In [12]:
class BERT(nn.Module):
    def __init__(self, vocab_size, max_len):
        super().__init__()

        d_model = 256
        n_heads = 4
        n_layers = 4

        self.embed = Embedding(vocab_size, max_len, d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads) for _ in range(n_layers)])

        self.mlm = nn.Linear(d_model, vocab_size)
        self.nsp = nn.Linear(d_model, 2)

    def forward(self, input_ids, segment_ids, masked_pos):

        x = self.embed(input_ids, segment_ids)

        mask = get_attn_pad_mask(input_ids)

        for layer in self.layers:
            x = layer(x, mask)

        cls = x[:,0]
        logits_nsp = self.nsp(cls)

        masked_pos = masked_pos[:,:,None].expand(-1,-1,x.size(-1))
        masked_output = torch.gather(x, 1, masked_pos)

        logits_mlm = self.mlm(masked_output)

        return logits_mlm, logits_nsp


## TRAINING

The BERT model was trained using the combined loss of Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).

Hyperparameters:

• Optimizer: Adam  
• Learning rate: 1e-4  
• Loss function: CrossEntropyLoss  
• Epochs: 400  
• Batch size: 6  
• Maximum sequence length: 128  
• Maximum masked tokens per sequence: 5  

At each epoch, a new batch is dynamically generated with random sentence pairs and masked tokens.




In [13]:
model = BERT(vocab_size, max_len).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 400

for epoch in range(num_epochs):

    batch = make_batch()
    input_ids, seg_ids, masked_tokens, masked_pos, isNext = \
        map(torch.LongTensor, zip(*batch))

    input_ids = input_ids.to(device)
    seg_ids = seg_ids.to(device)
    masked_tokens = masked_tokens.to(device)
    masked_pos = masked_pos.to(device)
    isNext = isNext.to(device)

    optimizer.zero_grad()

    logits_mlm, logits_nsp = model(input_ids, seg_ids, masked_pos)

    loss_mlm = criterion(logits_mlm.transpose(1,2), masked_tokens)
    loss_nsp = criterion(logits_nsp, isNext)

    loss = loss_mlm + loss_nsp

    if epoch % 20 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

    loss.backward()
    optimizer.step()


Epoch 0 Loss 11.8418
Epoch 20 Loss 7.8475
Epoch 40 Loss 8.8528
Epoch 60 Loss 6.0164
Epoch 80 Loss 3.7788
Epoch 100 Loss 6.8036
Epoch 120 Loss 6.6769
Epoch 140 Loss 6.2061
Epoch 160 Loss 6.0004
Epoch 180 Loss 5.5246
Epoch 200 Loss 4.5119
Epoch 220 Loss 5.4036
Epoch 240 Loss 6.0868
Epoch 260 Loss 5.2610
Epoch 280 Loss 4.9646
Epoch 300 Loss 4.9868
Epoch 320 Loss 6.9694
Epoch 340 Loss 4.2131
Epoch 360 Loss 6.8517
Epoch 380 Loss 5.7139


## SAVING

In [14]:
torch.save(model.state_dict(), "bert_task1_weights.pth")
print("Model saved for Task 2")


Model saved for Task 2


In [46]:
torch.save(word2id, "word2id.pth")



# TASK 2

## Loading Dataset

For Task 2, the Stanford Natural Language Inference (SNLI) dataset was used to train the Sentence-BERT model for the Natural Language Inference (NLI) task.

Dataset source:
Bowman et al., 2015 — SNLI Corpus  
Available via Hugging Face Datasets Library:
https://huggingface.co/datasets/snli

The SNLI dataset contains sentence pairs labeled as:

• Entailment  
• Neutral  
• Contradiction  

These labels indicate the semantic relationship between a premise and a hypothesis sentence.



In [17]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

snli = load_dataset("snli")

# remove invalid labels (-1)
snli = snli.filter(lambda x: x["label"] != -1)

train_data = snli["train"].shuffle(seed=42).select(range(5000))
val_data   = snli["validation"].shuffle(seed=42).select(range(1000))


## TOKENIZATION USING TASK-1 VOCAB

In [21]:
def encode_sentence(sentence):
    words = sentence.lower().split()
    ids = [word2id.get(w, 0) for w in words]  # unknown → PAD
    ids = ids[:128] + [0]*(128-len(ids))
    return ids


## PREPROCESS DATA 

1. The dataset was loaded using the Hugging Face `datasets` library.
2. Invalid samples with label `-1` (missing annotations) were removed.
3. A subset was selected to reduce computational cost:
   • 5,000 samples for training  
   • 1,000 samples for validation
4. The data was shuffled to ensure random sampling.

This dataset is suitable for training Sentence-BERT because it provides supervised labels for semantic relationships between sentences.

In [22]:
def preprocess(batch):
    return {
        "premise_ids": encode_sentence(batch["premise"]),
        "hypothesis_ids": encode_sentence(batch["hypothesis"]),
        "label": batch["label"]
    }

train_data = train_data.map(preprocess)
val_data   = val_data.map(preprocess)

train_data.set_format(type="torch",
    columns=["premise_ids","hypothesis_ids","label"])

val_data.set_format(type="torch",
    columns=["premise_ids","hypothesis_ids","label"])


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## DATALOADER

In [25]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32)




## Loading Pre-trained BERT Encoder from Task 1

The BERT model trained in Task 1 is loaded and reused as the encoder for the Sentence-BERT architecture.

• The saved model weights are loaded from disk.  
• The encoder is moved to the available device (GPU/CPU).  
• The model is set to training mode to allow fine-tuning during Task 2.

Reusing the pre-trained encoder enables transfer learning, allowing the model to leverage linguistic knowledge learned from the large unlabeled corpus in Task 1.


In [ ]:
bert = BERT(vocab_size, max_len).to(device) 

bert.load_state_dict(
    torch.load("bert_task1_weights.pth", weights_only=True)
)

bert.train()


BERT(
  (embed): Embedding(
    (tok): Embedding(43826, 256)
    (pos): Embedding(128, 256)
    (seg): Embedding(2, 256)
    (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_Q): Linear(in_features=256, out_features=256, bias=True)
        (W_K): Linear(in_features=256, out_features=256, bias=True)
        (W_V): Linear(in_features=256, out_features=256, bias=True)
        (fc): Linear(in_features=256, out_features=256, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
    )
  )
  (mlm): Linear(in_features=256, out_features=43826, bias=True)
  (nsp): Linear(in_features=256, out_features=2, bias=True)
)

## ATTENTION MASK FUNCTION

In [32]:
def get_attn_pad_mask(seq):
    return seq.eq(0).unsqueeze(1).expand(
        seq.size(0),
        seq.size(1),
        seq.size(1)
    )


## MEAN POOLING (SBERT)

In [33]:
def mean_pool(token_embeddings, input_ids):
    mask = (input_ids != 0).unsqueeze(-1)
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1)
    return summed / counts


## SENTENCE-BERT SIAMESE MODEL

In [34]:
class SentenceBERT(nn.Module):
    def __init__(self, bert, hidden_dim):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(hidden_dim*3, 3)

    def encode(self, input_ids):

        segment_ids = torch.zeros_like(input_ids).to(device)

        # Embedding
        x = self.bert.embed(input_ids, segment_ids)

        # Attention mask
        mask = get_attn_pad_mask(input_ids)

        # Encoder layers (fine-tuned)
        for layer in self.bert.layers:
            x = layer(x, mask)

        # SBERT pooling
        return mean_pool(x, input_ids)

    def forward(self, a_ids, b_ids):

        u = self.encode(a_ids)
        v = self.encode(b_ids)

        uv = torch.abs(u - v)

        x = torch.cat([u, v, uv], dim=-1)

        return self.classifier(x)


## TRAINING (SOFTMAX LOSS)

In [35]:
model = SentenceBERT(bert, 256).to(device)

optimizer = optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

epochs = 5


In [36]:
for epoch in range(epochs):

    # ===== TRAINING =====
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader):

        a_ids = batch["premise_ids"].to(device)
        b_ids = batch["hypothesis_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(a_ids, b_ids)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:

            a_ids = batch["premise_ids"].to(device)
            b_ids = batch["hypothesis_ids"].to(device)
            labels = batch["label"].to(device)

            outputs = model(a_ids, b_ids)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}")


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch 1 Train Loss: 1.0752
Epoch 1 Val Loss: 1.0540


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch 2 Train Loss: 1.0382
Epoch 2 Val Loss: 1.0326


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch 3 Train Loss: 1.0082
Epoch 3 Val Loss: 1.0366


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch 4 Train Loss: 0.9831
Epoch 4 Val Loss: 1.0045


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch 5 Train Loss: 0.9580
Epoch 5 Val Loss: 1.0053


### Training and Validation of Sentence-BERT (Fine-tuning)

Training Procedure:
• The model was trained for multiple epochs using mini-batches.
• Cross-entropy loss was used for the NLI classification task.
• Gradients were computed via backpropagation and parameters updated using Adam optimizer.

Validation:
• After each epoch, the model was evaluated on a held-out validation set.
• Validation loss was computed without gradient updates to monitor generalization performance.

Results:
The training loss steadily decreased across epochs, indicating successful learning.
Validation loss also showed improvement, suggesting the model was able to generalize to unseen data.

Example Loss Progression:
Epoch 1 — Train Loss: 1.0752 | Val Loss: 1.0540  
Epoch 5 — Train Loss: 0.9580 | Val Loss: 1.0053  

Overall, the decreasing trend demonstrates that the fine-tuned encoder learned meaningful semantic representations for the NLI task.


## SAVING 

In [37]:
torch.save(model.state_dict(), "sbert_finetuned_model.pth")

print("SBERT fine-tuned model saved")


SBERT fine-tuned model saved


## SENTENCE EMBEDDING + COSINE SIMILARITY

In [38]:
from torch.nn.functional import cosine_similarity

model.eval()

def sentence_embedding(sentence):
    ids = torch.tensor([encode_sentence(sentence)]).to(device)
    return model.encode(ids)

s1 = "A man is eating food"
s2 = "A person eats a meal"

e1 = sentence_embedding(s1)
e2 = sentence_embedding(s2)

sim = cosine_similarity(e1, e2).item()

print("Cosine similarity:", sim)


Cosine similarity: 0.9319499731063843


### Cosine Similarity

The cosine similarity between the two example sentences is **0.93**, indicating a high semantic similarity. This demonstrates that the trained Sentence-BERT model successfully captures semantic meaning in sentence embeddings.


# TASK 3 

### NLI Classification Performance

The model achieved an overall accuracy of **49%** on the SNLI validation subset. Performance across classes is moderate, with the highest recall observed for the neutral class. This indicates that the model learned meaningful semantic relationships but is limited by reduced training data and model size.


In [43]:
from sklearn.metrics import classification_report
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:

        a_ids = batch["premise_ids"].to(device)
        b_ids = batch["hypothesis_ids"].to(device)
        labels = batch["label"].to(device)

        outputs = model(a_ids, b_ids)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# CLASSIFICATION REPORT
print(classification_report(
    all_labels,
    all_preds,
    target_names=["entailment", "neutral", "contradiction"]
))

               precision    recall  f1-score   support

   entailment       0.56      0.43      0.48       374
      neutral       0.45      0.60      0.52       294
contradiction       0.46      0.45      0.46       332

     accuracy                           0.49      1000
    macro avg       0.49      0.49      0.49      1000
 weighted avg       0.50      0.49      0.48      1000



### Datasets Used:
• BookCorpus (subset of 100K samples) — for BERT pretraining  
• SNLI dataset — for Sentence-BERT fine-tuning and evaluation  

Key Hyperparameters:

• Hidden size: 256  
• Encoder layers: 4  
• Attention heads: 4  
• Maximum sequence length: 128  
• Batch size: small due to memory limits  
• Optimizer: Adam  

Modifications from Original Models:
• Mini-BERT architecture for feasibility  
• Custom tokenizer instead of WordPiece  
• Reduced training data  
• Mean pooling for sentence embeddings  #



### Limitations and Challenges

1. Limited Training Data
Due to computational constraints, only a subset of the BookCorpus (100K samples) and SNLI dataset (5K training pairs) was used. This limited the model’s ability to learn rich linguistic patterns compared to training on full-scale corpora used in the original BERT and SBERT papers.

2. Simplified Tokenization
A word-level tokenizer built from scratch was used instead of the original WordPiece tokenizer. This resulted in a larger vocabulary, lack of subword handling, and reduced ability to generalize to unseen words.

3. Reduced Model Size (Mini-BERT)
The implemented BERT used smaller hyperparameters (d_model = 256, 4 layers, 4 attention heads) compared to BERT-base (768 hidden size, 12 layers, 12 heads). While necessary for GPU memory constraints, this reduced representational power.

4. Training Stability
Training BERT from scratch is challenging and sensitive to hyperparameters. Loss fluctuations were observed due to small batch size and limited data, which can affect convergence.

5. Computational Constraints
GPU memory limitations required:
• Smaller batch sizes  
• Fewer epochs  
• Reduced model depth  

These constraints impacted overall performance.

6. Evaluation Limitations
The model achieved moderate accuracy on the SNLI validation set, indicating that semantic understanding was learned but not at state-of-the-art level.

---

### Potential Improvements

1. Use Larger Datasets
Training on full BookCorpus and Wikipedia datasets would significantly improve language understanding and embedding quality.

2. Implement Subword Tokenization
Replacing word-level tokenization with WordPiece or Byte-Pair Encoding (BPE) would:
• Reduce vocabulary size  
• Handle unknown words better  
• Improve generalization  

3. Increase Model Capacity
Using a larger architecture closer to BERT-base would improve performance if sufficient computational resources are available.

4. Longer Training
Increasing training epochs and dataset size would allow the model to converge more effectively.

5. Fine-Tuning Strategies
Experimenting with:
• Learning rate scheduling  
• Layer-wise learning rates  
• Gradual unfreezing  

could improve Sentence-BERT performance.

6. Advanced Pooling Methods
Using CLS-token pooling or attention-based pooling instead of simple mean pooling may produce stronger sentence embeddings.

7. Data Augmentation
Including additional NLI datasets (e.g., MNLI) would improve robustness and generalization.


